# SMS Spam Detection using NLP

This notebook demonstrates an end-to-end NLP pipeline for SMS Spam detection on the UCI SMS Spam Collection dataset.
Key design highlights:
- **Leakage-free Preprocessing & Modeling:** All TF-IDF transformations and classifiers are bundled in `sklearn.pipeline.Pipeline`.
- **Group-Aware Evaluation:** Both the hold-out split and cross-validation use `StratifiedGroupKFold` keyed on raw SMS text to prevent identical repeated messages from crossing evaluation boundaries.
- **Calibrated Probabilities:** The Linear SVM model is calibrated with `CalibratedClassifierCV` to output reliable spam probabilities.


In [ ]:
import sys
import os
import csv
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Ensure repository root is on Python path
sys.path.insert(0, os.path.abspath('..'))
from src.preprocessing import clean_text

from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

## 1. Load Data
We read the raw dataset using `quoting=csv.QUOTE_NONE` to correctly parse all 5,574 messages without dropping records due to unescaped quotes.

In [ ]:
data_path = '../data/SMSSpamCollection'
df = pd.read_csv(data_path, sep='\t', names=["label", "message"], quoting=csv.QUOTE_NONE)
print(f"Total records: {len(df)}")
print(f"Class distribution:\n{df['label'].value_counts()}\n")
df.head()

## 2. Text Preprocessing
We map the target labels (`ham` -> 0, `spam` -> 1) and apply `clean_text` from `src.preprocessing`.

In [ ]:
df['clean_message'] = df['message'].apply(clean_text)
df['target'] = df['label'].map({"ham": 0, "spam": 1})
df[['label', 'message', 'clean_message']].head()

## 3. Group-Aware Hold-Out Split
We use `StratifiedGroupKFold` on the raw message string to ensure duplicate messages never cross between train and test partitions.

In [ ]:
groups = df["message"]
holdout_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

train_idx, test_idx = next(holdout_cv.split(df["clean_message"], df["target"], groups=groups))

X_train = df.iloc[train_idx]["clean_message"]
X_test = df.iloc[test_idx]["clean_message"]
y_train = df.iloc[train_idx]["target"]
y_test = df.iloc[test_idx]["target"]
groups_train = groups.iloc[train_idx]

overlap = set(groups.iloc[train_idx]).intersection(set(groups.iloc[test_idx]))
assert len(overlap) == 0, f"Leakage detected: {len(overlap)} groups overlap!"

print(f"Train samples: {len(X_train)} (Spam: {y_train.sum()}, Ham: {(y_train==0).sum()})")
print(f"Test samples:  {len(X_test)} (Spam: {y_test.sum()}, Ham: {(y_test==0).sum()})")
print(f"Verified: 0 duplicate messages cross the train/test split boundary.")

## 4. Pipeline Definition & Model Benchmarking (5-Fold Stratified Grouped CV)
We construct distinct pipelines for each candidate model and evaluate them solely on the training partition.

In [ ]:
def build_pipelines():
    def tfidf():
        return TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
    
    return {
        "Multinomial Naive Bayes": Pipeline([("tfidf", tfidf()), ("clf", MultinomialNB())]),
        "Logistic Regression": Pipeline([("tfidf", tfidf()), ("clf", LogisticRegression(max_iter=1000, random_state=42))]),
        "Linear SVM": Pipeline([("tfidf", tfidf()), ("clf", CalibratedClassifierCV(LinearSVC(random_state=42, dual="auto")))]),
        "Random Forest": Pipeline([("tfidf", tfidf()), ("clf", RandomForestClassifier(n_estimators=100, random_state=42))])
    }

candidate_models = build_pipelines()
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

best_model_name = None
best_cv_f1 = -1.0
cv_summary = {}

print(f"{'Model':<25} | {'CV Accuracy':<12} | {'CV Precision':<12} | {'CV Recall':<12} | {'CV F1':<10}")
print("-" * 75)

for name, pipeline in candidate_models.items():
    cv_res = cross_validate(
        pipeline, 
        X_train, 
        y_train, 
        groups=groups_train, 
        cv=cv, 
        scoring=['accuracy', 'precision', 'recall', 'f1'],
        n_jobs=-1
    )
    mean_acc = cv_res['test_accuracy'].mean()
    mean_prec = cv_res['test_precision'].mean()
    mean_rec = cv_res['test_recall'].mean()
    mean_f1 = cv_res['test_f1'].mean()
    
    cv_summary[name] = {"Accuracy": mean_acc, "Precision": mean_prec, "Recall": mean_rec, "F1": mean_f1}
    print(f"{name:<25} | {mean_acc:<12.4f} | {mean_prec:<12.4f} | {mean_rec:<12.4f} | {mean_f1:<10.4f}")
    
    if mean_f1 > best_cv_f1:
        best_cv_f1 = mean_f1
        best_model_name = name

print("-" * 75)
print(f"Selected Best Model: {best_model_name} (Grouped CV F1: {best_cv_f1:.4f})")

## 5. Final Evaluation on Hold-Out Test Set
We fit the winning pipeline on the full training partition and evaluate it on the unseen test set.

In [ ]:
best_pipeline = build_pipelines()[best_model_name]
best_pipeline.fit(X_train, y_train)

y_pred = best_pipeline.predict(X_test)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]

print("Classification Report on Unseen Hold-out Test Set:\n")
print(classification_report(y_test, y_pred, target_names=["ham", "spam"], digits=4))
print(f"ROC AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['ham', 'spam'], yticklabels=['ham', 'spam'])
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.title(f'Confusion Matrix - {best_model_name} (Hold-out Test Set)')
plt.show()